In [7]:
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import spacy
from spacy import displacy
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\usama\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
bbc_news = pd.read_csv('111 - bbc-news.csv')
bbc_news.head()

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?a...,With much of the UK enduring another period of...
1,1,9267,'Liz Truss the Brief?' World reacts to UK poli...,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_m...,The UK's political chaos has been watched arou...
2,2,7387,Rationing energy is nothing new for off-grid c...,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlan...,https://www.bbc.co.uk/news/uk-scotland-highlan...,Scoraig in the north west Highlands has long h...
3,3,767,The hunt for superyachts of sanctioned Russian...,"Tue, 22 Mar 2022 14:37:01 GMT",https://www.bbc.co.uk/news/60739336,https://www.bbc.co.uk/news/60739336?at_medium=...,"Wealthy Russians sanctioned by the US, EU and ..."
4,4,3712,Platinum Jubilee: 70 years of the Queen in 70 ...,"Wed, 01 Jun 2022 23:17:33 GMT",https://www.bbc.co.uk/news/uk-61660128,https://www.bbc.co.uk/news/uk-61660128?at_medi...,A quick look back at the Queen's 70 years on t...


In [9]:
bbc_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   1000 non-null   int64 
 1   index        1000 non-null   int64 
 2   title        1000 non-null   object
 3   pubDate      1000 non-null   object
 4   guid         1000 non-null   object
 5   link         1000 non-null   object
 6   description  1000 non-null   object
dtypes: int64(2), object(5)
memory usage: 54.8+ KB


In [10]:
titles = pd.DataFrame(bbc_news['title'])
titles.head()

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...
2,Rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...


In [11]:
titles['lowercase'] = titles['title'].str.lower()
titles.head()

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...


In [12]:
en_stopwords = stopwords.words('english')
print(en_stopwords)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [13]:
titles['stopwords'] = titles['lowercase'].apply(lambda x: ' '.join(word for word in x.split() if word not in en_stopwords))
titles.head()

,title,lowercase,stopwords
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds


In [14]:
titles['no_punc'] = titles['stopwords'].apply(lambda x: re.sub('[^\w\s]','',x))
titles.head()

,title,lowercase,stopwords,no_punc
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds


In [15]:
titles['token_clean'] = titles['no_punc'].apply(lambda x: word_tokenize(x))
titles.head()

,title,lowercase,stopwords,no_punc,token_clean
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[liz, truss, brief, world, reacts, uk, politic..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[rationing, energy, nothing, new, offgrid, com..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[platinum, jubilee, 70, years, queen, 70, seco..."


In [16]:
titles['token_raw'] = titles['title'].apply(lambda x: word_tokenize(x))
titles.head()

,title,lowercase,stopwords,no_punc,token_clean,token_raw
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[refuse, work]","[Can, I, refuse, to, work, ?]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[liz, truss, brief, world, reacts, uk, politic...","['Liz, Truss, the, Brief, ?, ', World, reacts,..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[rationing, energy, nothing, new, offgrid, com...","[Rationing, energy, is, nothing, new, for, off..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[hunt, superyachts, sanctioned, russian, oliga...","[The, hunt, for, superyachts, of, sanctioned, ..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[platinum, jubilee, 70, years, queen, 70, seco...","[Platinum, Jubilee, :, 70, years, of, the, Que..."


In [17]:
lemmatizer = WordNetLemmatizer()
titles["token_clean_lemmatize"] = titles["token_clean"].apply(lambda x: [lemmatizer.lemmatize(token) for token in x])
titles.head()

,title,lowercase,stopwords,no_punc,token_clean,token_raw,token_clean_lemmatize
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[refuse, work]","[Can, I, refuse, to, work, ?]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK poli...,'liz truss the brief?' world reacts to uk poli...,'liz truss brief?' world reacts uk political t...,liz truss brief world reacts uk political turmoil,"[liz, truss, brief, world, reacts, uk, politic...","['Liz, Truss, the, Brief, ?, ', World, reacts,...","[liz, truss, brief, world, reacts, uk, politic..."
2,Rationing energy is nothing new for off-grid c...,rationing energy is nothing new for off-grid c...,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[rationing, energy, nothing, new, offgrid, com...","[Rationing, energy, is, nothing, new, for, off...","[rationing, energy, nothing, new, offgrid, com..."
3,The hunt for superyachts of sanctioned Russian...,the hunt for superyachts of sanctioned russian...,hunt superyachts sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs,"[hunt, superyachts, sanctioned, russian, oliga...","[The, hunt, for, superyachts, of, sanctioned, ...","[hunt, superyachts, sanctioned, russian, oliga..."
4,Platinum Jubilee: 70 years of the Queen in 70 ...,platinum jubilee: 70 years of the queen in 70 ...,platinum jubilee: 70 years queen 70 seconds,platinum jubilee 70 years queen 70 seconds,"[platinum, jubilee, 70, years, queen, 70, seco...","[Platinum, Jubilee, :, 70, years, of, the, Que...","[platinum, jubilee, 70, year, queen, 70, second]"


In [18]:
list_raw = sum(titles["token_raw"],[])
list_clean = sum(titles["token_clean_lemmatize"],[])

In [19]:
nlp = spacy.load("en_core_web_sm")

In [20]:
spacy_doc_clean = nlp(' '.join(list_clean))
spacy_doc_clean

refuse work liz truss brief world reacts uk political turmoil rationing energy nothing new offgrid community hunt superyachts sanctioned russian oligarch platinum jubilee 70 year queen 70 second red bull found guilty breaking formula 1 budget cap world triathlon championship series flora duffy beat georgia taylorbrown womens title terry hall coventry scooter rideout pay tribute singer post office fujitsu face inquiry horizon scandal pavement parking frightens me uk interest rate rise affect high could go stayed storm happens now six nation scotland best since 99 beat best ireland ever long liz truss survive prime minister platinum jubilee beacon light across globe celebrate queen reign top student loan interest rate cut 5 england russian said theyd take baby medic story week picture 1117 june 2022 xi jinping hong kong tight security city mark 25 year handover rescue carried silence listen sound life nick pope newcastle goalkeeper extends incredible clean sheet record china plan become 

In [21]:
pos_bbc = pd.DataFrame(columns = ["token", "pos"])

for word in spacy_doc_clean:
    pos_bbc = pd.concat([pos_bbc,pd.DataFrame.from_records([{"token" : word.text, "pos" : word.pos_}])],ignore_index =True)

pos_bbc.head()

,token,pos
0,refuse,AUX
1,work,NOUN
2,liz,PROPN
3,truss,ADJ
4,brief,ADJ


In [22]:
pos_bbc_count = pos_bbc.groupby(["token","pos"]).size().reset_index( name = "count").sort_values(by = "count", ascending = False)
pos_bbc_count.head()

,token,pos,count
30,2022,NUM,47
1162,england,PROPN,45
870,cup,PROPN,39
3056,say,VERB,37
3707,uk,PROPN,37


In [23]:
pos_bbc_noun = pos_bbc_count[pos_bbc_count.pos == "NOUN"][0:10]
pos_bbc_noun

,token,pos,count
3840,war,NOUN,34
3948,world,NOUN,30
2136,man,NOUN,22
907,day,NOUN,21
3973,year,NOUN,20
1158,energy,NOUN,17
2847,record,NOUN,17
2657,police,NOUN,16
3870,week,NOUN,16
3935,woman,NOUN,16


In [24]:
pos_bbc_verb = pos_bbc_count[pos_bbc_count.pos == "VERB"][0:10]
pos_bbc_verb

,token,pos,count
3056,say,VERB,37
3711,ukraine,VERB,22
1380,found,VERB,13
3461,take,VERB,13
1651,hit,VERB,13
358,beat,VERB,13
2133,make,VERB,13
1459,get,VERB,13
1473,give,VERB,11
3918,win,VERB,10


In [25]:
spacy_doc = nlp(' '.join(list_raw))
spacy_doc

Can I refuse to work ? 'Liz Truss the Brief ? ' World reacts to UK political turmoil Rationing energy is nothing new for off-grid community The hunt for superyachts of sanctioned Russian oligarchs Platinum Jubilee : 70 years of the Queen in 70 seconds Red Bull found guilty of breaking Formula 1 's budget cap World Triathlon Championship Series : Flora Duffy beats Georgia Taylor-Brown to women 's title Terry Hall : Coventry scooter ride-out pays tribute to singer Post Office and Fujitsu to face inquiry over Horizon scandal 'Pavement parking frightens me ' UK interest rates : How will the rise affect you and how high could it go ? They stayed for the storm - what happens now ? Six Nations : Can Scotland 's best since '99 beat best Ireland ever ? How long can Liz Truss survive as prime minister ? Platinum Jubilee : Beacons light across the globe to celebrate Queen 's reign Top student loan interest rate cut by 5 % in England Russians said they ’ d take my baby : A medic ’ s story Week in 

In [26]:
pd_Ner = pd.DataFrame(columns = ["token", "NER_tag"])

for word in spacy_doc.ents:
    if pd.isna(word.label_) is False:
        pd_Ner = pd.concat([pd_Ner,pd.DataFrame.from_records([{"token" : word.text, "NER_tag" : word.label_}])],ignore_index= True)

pd_Ner.head()

,token,NER_tag
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP


In [27]:
displacy.render(spacy_doc, style = "ent", jupyter = True)

In [28]:
pd_token_count = pd_Ner.groupby(["token","NER_tag"]).size().reset_index(name = "count").sort_values(by = "count", ascending = False)
pd_token_count.head()

,token,NER_tag,count
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19


In [40]:
pd_Ner_count = pd_token_count.groupby(['NER_tag'])['token'].count().sort_values(  ascending = False)
pd_Ner_count.head()

NER_tag
PERSON      376
ORG         276
GPE         134
DATE         74
CARDINAL     62
Name: token, dtype: int64

In [45]:
pd_Ner_Person = pd_token_count[pd_token_count.NER_tag == 'PERSON']
pd_Ner_Person.head()

,token,NER_tag,count
257,Covid,PERSON,9
757,Putin,PERSON,8
760,Queen,PERSON,8
563,Liz Truss,PERSON,6
169,Boris Johnson,PERSON,6
